In [0]:
from pyspark.sql import functions as F

data = [
    (1, "Alice", 120.0, "2026-10-01"),
    (2, "Bob", 80.0, "2026-10-01"),
    (3, "Alice", 200.0, "2026-10-02"),
    (3, "Alice", 200.0, "2026-10-02"), # Duplicate
    (4, "Charlie", None, "2026-10-03"),
    (5, "Bob", 150.0, "2026-10-03"),
    (6, "Ope", 120.0, "2026-10-01"),
    (7, "Chbka", 120.0, "2026-10-01"),

]

columns = ["order_id", "customer", "amount", "order_date"]

df = spark.createDataFrame(
    data,
    schema=columns
)

display(df)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_orders")

In [0]:
%sql

--DESCRIBE DETAIL bronze_orders;
SELECT * FROM bronze_orders 
VERSION AS OF 0

In [0]:
silver_df = (
    spark.table("bronze_orders")
    .dropDuplicates(["order_id"])
    .filter(F.col("amount").isNotNull())
    .withColumn("order_date", F.to_date("order_date"))
)

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders")


In [0]:
%sql

SELECT *
FROM silver_orders;